# 05 — Final Inference

"
        "Load the saved physics-aware CO₂ model and use the deployable "
        "soft regime gate to predict compressibility and pressure from "
        "temperature and molar density.

"
        "Inputs use `T_K` in kelvin and `rho_mol_m3` in mol/m³.


In [ ]:
!pip install -q pysr scikit-learn CoolProp


## Restore saved artifacts in Colab

"
        "Run this cell in a fresh Colab runtime before loading the models. "
        "It copies the persistent Drive folders into `/content`.


In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_root = Path(
        "/content/drive/MyDrive/ideal-gas-correction-model"
    )

    shutil.copytree(
        drive_root / "data",
        "/content/data",
        dirs_exist_ok=True
    )
    shutil.copytree(
        drive_root / "results",
        "/content/results",
        dirs_exist_ok=True
    )

    print("Restored data and results from Google Drive.")
except ImportError:
    print("Not running in Colab; using local project files.")


In [ ]:
import json
import pickle

import CoolProp.CoolProp as CP
import numpy as np
import pandas as pd
from pysr import PySRRegressor


def find_project_root():
    candidates = [
        Path("/content"),
        Path.cwd(),
        Path.cwd().parent
    ]

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate

    return Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Processed data directory not found: {DATA_DIR}"
    )

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model directory not found: {MODEL_DIR}"
    )

TABLE_DIR.mkdir(parents=True, exist_ok=True)

with open(MODEL_DIR / "final_regime_models.pkl", "rb") as file:
    final_regime_models = pickle.load(file)

with open(MODEL_DIR / "final_regime_indices.json", "r") as file:
    final_regime_indices = {
        name: int(index)
        for name, index in json.load(file).items()
    }

with open(MODEL_DIR / "final_gate.pkl", "rb") as file:
    final_gate = pickle.load(file)

with open(MODEL_DIR / "gate_config.json", "r") as file:
    gate_config = json.load(file)

R = 8.31446261815324
FLUID = "CO2"
T_CRITICAL = CP.PropsSI("Tcrit", FLUID)
RHO_CRITICAL = CP.PropsSI("rhomolar_critical", FLUID)
FEATURE_COLUMNS = ["T_reduced", "rho_reduced"]
REGIME_NAMES = gate_config["regime_names"]

print("Project root:", PROJECT_ROOT)
print("Routing method:", gate_config["routing_method"])
print("Final gate depth:", gate_config["final_gate_depth"])


## Deployable prediction function

"
        "The final model uses probability-weighted predictions from the three "
        "regime-specific symbolic models. The displayed regime is the gate's "
        "highest-probability class; the pressure prediction still uses the "
        "soft probability mixture.


In [ ]:
def predict_co2(T_K, rho_mol_m3):
    T_array, rho_array = np.broadcast_arrays(
        np.asarray(T_K, dtype=float),
        np.asarray(rho_mol_m3, dtype=float)
    )

    if np.any(T_array <= 0) or np.any(rho_array <= 0):
        raise ValueError("T_K and rho_mol_m3 must be positive.")

    result = pd.DataFrame({
        "T_K": T_array.ravel(),
        "rho_mol_m3": rho_array.ravel()
    })

    result["T_reduced"] = result["T_K"] / T_CRITICAL
    result["rho_reduced"] = (
        result["rho_mol_m3"] / RHO_CRITICAL
    )
    result["p_ideal_Pa"] = (
        result["rho_mol_m3"]
        * R
        * result["T_K"]
    )

    X = result[FEATURE_COLUMNS].to_numpy()
    gate_features = result[gate_config["gate_features"]]
    gate_probabilities = final_gate.predict_proba(gate_features)

    regime_predictions = np.column_stack([
        final_regime_models[regime].predict(
            X,
            index=final_regime_indices[regime]
        )
        for regime in REGIME_NAMES
    ])

    ordered_probabilities = np.column_stack([
        gate_probabilities[:, list(final_gate.classes_).index(regime)]
        for regime in REGIME_NAMES
    ])

    predicted_delta_z = np.sum(
        ordered_probabilities * regime_predictions,
        axis=1
    )

    result["predicted_regime"] = final_gate.predict(gate_features)

    for index, regime in enumerate(REGIME_NAMES):
        result[f"gate_probability_{regime}"] = (
            ordered_probabilities[:, index]
        )

    result["delta_Z_prediction"] = predicted_delta_z
    result["Z_prediction"] = 1.0 + predicted_delta_z
    result["p_prediction_Pa"] = (
        result["p_ideal_Pa"] * result["Z_prediction"]
    )
    result["p_prediction_MPa"] = (
        result["p_prediction_Pa"] / 1_000_000.0
    )

    return result


In [ ]:
test_prediction = predict_co2(
    T_K=[400.0, 450.0],
    rho_mol_m3=[10000.0, 15000.0]
)

display(test_prediction)


## Compare with CoolProp


In [ ]:
comparison = test_prediction.copy()

comparison["p_coolprop_Pa"] = [
    CP.PropsSI(
        "P",
        "T", float(T),
        "Dmolar", float(rho),
        FLUID
    )
    for T, rho in zip(
        comparison["T_K"],
        comparison["rho_mol_m3"]
    )
]

comparison["Z_coolprop"] = (
    comparison["p_coolprop_Pa"]
    / comparison["p_ideal_Pa"]
)

comparison["pressure_error_percent"] = (
    abs(
        comparison["p_prediction_Pa"]
        - comparison["p_coolprop_Pa"]
    )
    / comparison["p_coolprop_Pa"]
    * 100.0
)

display(
    comparison[
        [
            "T_K",
            "rho_mol_m3",
            "Z_prediction",
            "Z_coolprop",
            "p_prediction_MPa",
            "p_coolprop_Pa",
            "pressure_error_percent"
        ]
    ]
)


## Twenty-point stress sanity check

"
        "This is an interface check after reloading the serialized artifacts. "
        "The full held-out test metrics remain the primary evaluation.


In [ ]:
stress_test_df = pd.read_csv(
    DATA_DIR / "co2_stress_test.csv"
)

sample = stress_test_df.sample(
    n=min(20, len(stress_test_df)),
    random_state=47
).reset_index(drop=True)

batch_check = predict_co2(
    sample["T_K"].to_numpy(),
    sample["rho_mol_m3"].to_numpy()
).reset_index(drop=True)

batch_check["p_coolprop_Pa"] = (
    sample["p_real_Pa"].to_numpy()
)

batch_check["pressure_error_percent"] = (
    abs(
        batch_check["p_prediction_Pa"]
        - batch_check["p_coolprop_Pa"]
    )
    / batch_check["p_coolprop_Pa"]
    * 100.0
)

print(
    "Mean pressure error:",
    batch_check["pressure_error_percent"].mean(),
    "%"
)
print(
    "Maximum pressure error:",
    batch_check["pressure_error_percent"].max(),
    "%"
)

display(
    batch_check.sort_values(
        "pressure_error_percent",
        ascending=False
    ).head(10)
)


In [ ]:
batch_check.to_csv(
    TABLE_DIR / "inference_sanity_check_20_points.csv",
    index=False
)

test_prediction.to_csv(
    TABLE_DIR / "inference_example_predictions.csv",
    index=False
)

print("Inference tables saved to:", TABLE_DIR)


## Optional Drive backup

"
        "Run this after saving the tables if the notebook is running in Colab.


In [ ]:
drive_root = Path(
    "/content/drive/MyDrive/ideal-gas-correction-model"
)

if drive_root.exists():
    shutil.copytree(
        RESULTS_DIR,
        drive_root / "results",
        dirs_exist_ok=True
    )
    print("Inference results backed up to Drive.")
else:
    print("Drive folder not found; local tables are still saved.")
